In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8209.01it/s]


In [5]:
input_text = "The future of AI is"

inputs = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=30, repetition_penalty=2.0)

print(tokenizer.decode(outputs[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The future of AI is uncertain. The question now becomes whether we can make it work in a way that makes sense to humans, or if the technology will be too complex and


In [3]:
outputs = model(**inputs)

print(outputs.logits.shape)

torch.Size([1, 5, 50257])


In [6]:
input_text="The meaning of life is"
"Machine learning will"
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=30, repetition_penalty=2.0)
print(tokenizer.decode(outputs[0]))
outputs = model(**inputs)
print(outputs.logits.shape)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The meaning of life is not the same as that which we have been taught.
I am a Christian, and I believe in God's Word; but my faith has always
torch.Size([1, 5, 50257])


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8297.67it/s]


In [10]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=8,                      # rank (low-rank dimension)
    lora_alpha=16,            # scaling factor
    target_modules=["c_attn"], # where LoRA is applied
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [11]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


c:\Users\nagen\Desktop\ML&DLProjects\LLMs\llm_env\Lib\site-packages\peft\tuners\lora\layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [12]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

base_model.model.transformer.h.0.attn.c_attn.lora_A.default.weight
base_model.model.transformer.h.0.attn.c_attn.lora_B.default.weight
base_model.model.transformer.h.1.attn.c_attn.lora_A.default.weight
base_model.model.transformer.h.1.attn.c_attn.lora_B.default.weight
base_model.model.transformer.h.2.attn.c_attn.lora_A.default.weight
base_model.model.transformer.h.2.attn.c_attn.lora_B.default.weight
base_model.model.transformer.h.3.attn.c_attn.lora_A.default.weight
base_model.model.transformer.h.3.attn.c_attn.lora_B.default.weight
base_model.model.transformer.h.4.attn.c_attn.lora_A.default.weight
base_model.model.transformer.h.4.attn.c_attn.lora_B.default.weight
base_model.model.transformer.h.5.attn.c_attn.lora_A.default.weight
base_model.model.transformer.h.5.attn.c_attn.lora_B.default.weight
base_model.model.transformer.h.6.attn.c_attn.lora_A.default.weight
base_model.model.transformer.h.6.attn.c_attn.lora_B.default.weight
base_model.model.transformer.h.7.attn.c_attn.lora_A.default.we

In [15]:
input_text = "AI is transforming"

inputs = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=30, repetition_penalty=2.0)

print(tokenizer.decode(outputs[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


AI is transforming the way we think about technology. It's a new kind of thinking that has been around for decades, and it will be very interesting to see how


In [29]:
# training data
data = [
    {"text": "AI is the future of technology."},
    {"text": "Machine learning enables computers to learn from data."},
    {"text": "Deep learning is a subset of machine learning."},
    {"text": "Neural networks are inspired by the human brain."}
 ]
data=data*5
from datasets import Dataset

dataset = Dataset.from_list(data)

def tokenize_function(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=32)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 20/20 [00:00<00:00, 5391.48 examples/s]


In [30]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=20,
    logging_steps=1,
    save_strategy="no"
 )
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
 )

In [31]:
trainer.train()


c:\Users\nagen\Desktop\ML&DLProjects\LLMs\llm_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.021750
2,0.165333
3,0.042388
4,0.010659
5,0.176819
6,0.263684
7,0.094000
8,0.139472
9,0.003714
10,0.121111


TrainOutput(global_step=200, training_loss=0.0411848030645433, metrics={'train_runtime': 116.0389, 'train_samples_per_second': 3.447, 'train_steps_per_second': 1.724, 'total_flos': 6532300800000.0, 'train_loss': 0.0411848030645433, 'epoch': 20.0})

In [32]:
input_text = "AI is"

inputs = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=30, do_sample=False)

print(tokenizer.decode(outputs[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


AI is the future of technology.<|endoftext|>
